# Artificial Intelligence — Lab 7
## Constraint Satisfaction Problems: Modeling and Backtracking Search

**Course Learning Outcome — CLO4**  
Investigate Constraint Satisfaction Problems (CSPs), their representations, constraints, and solution methods.

**Environment:** Python 3 / Jupyter Notebook  
**Submission:** completed notebook containing predictions, traces, code, justifications, experiments, debugging answers, and reflection.

> **Assessment principle:** Correct code is only one part of the evidence. Most marks come from your ability to **model variables, domains, and constraints correctly; predict the effect of assignments; trace backtracking; justify consistency checks; and explain why a search branch succeeds or fails**.

## Lab at a Glance

| Stage | Suggested time | What you will do |
|---|---:|---|
| 1. CSP modeling | 15 min | Identify variables, domains, and constraints |
| 2. Manual consistency reasoning | 20 min | Predict legal assignments before coding |
| 3. Implement backtracking | 35 min | Build a generic recursive CSP solver |
| 4. Trace and analyze search | 20 min | Interpret assignments, failures, and backtracks |
| 5. Model a second CSP | 20 min | Apply the framework to a new problem |
| 6. Debugging, variation & reflection | 10 min | Diagnose errors and defend conclusions |

> **Main idea:** A CSP separates a problem into **variables**, **domains**, and **constraints**. Backtracking search constructs a solution incrementally and abandons partial assignments that violate constraints.

## Learning Objectives

By the end of this lab, you should be able to:

1. represent a CSP as

$$
(X,D,C);
$$

2. distinguish variables, domains, assignments, and constraints;
3. test whether a partial assignment is consistent;
4. manually trace backtracking search;
5. implement a generic recursive backtracking solver;
6. explain why early constraint checking reduces unnecessary search;
7. distinguish a complete assignment from a consistent partial assignment;
8. model a second real CSP using the same framework;
9. diagnose common CSP implementation errors;
10. justify how variable order can affect search effort.

In [ ]:
from typing import Dict, List, Set, Tuple, Optional, Callable
from copy import deepcopy

print("Lab 7 environment ready.")

# Part I — CSP Fundamentals

A Constraint Satisfaction Problem can be represented as

$$
(X,D,C)
$$

where:

- $X=\{X_1,X_2,\ldots,X_n\}$ is a set of **variables**;
- $D=\{D_1,D_2,\ldots,D_n\}$ gives a **domain** for each variable;
- $C$ is a set of **constraints** restricting legal combinations of values.

A solution is a **complete assignment** that satisfies every constraint.

## Task 1.1 — Identify the CSP Components

Suppose we want to schedule three exams:

- `AI`
- `DB`
- `Networks`

Available time slots are:

```text
Morning, Afternoon
```

Constraints:

1. `AI` and `DB` cannot be at the same time.
2. `DB` and `Networks` cannot be at the same time.

Answer:

- **Variables $X$:**  
- **Domains $D$:**  
- **Constraints $C$:**  

Then decide whether each assignment is valid.

| Assignment | Valid / Invalid | Justification |
|---|---|---|
| `AI=Morning, DB=Afternoon, Networks=Morning` |  |  |
| `AI=Morning, DB=Morning, Networks=Afternoon` |  |  |
| `AI=Afternoon, DB=Morning, Networks=Morning` |  |  |

**Your answers:**

## Task 1.2 — Partial vs. Complete Assignments

For the same exam CSP:

1. Is `{AI=Morning}` a complete assignment?
2. Can it still be consistent?
3. Why should a backtracking solver test consistency **before** all variables are assigned?
4. What advantage is gained by rejecting an inconsistent partial assignment immediately?

**Your answers:**

# Part II — Australia Map-Coloring CSP

We now use the classic Australia map-coloring problem.

Variables:

```text
WA, NT, SA, Q, NSW, V, T
```

Colors:

```text
Red, Green, Blue
```

Neighboring regions must have different colors.

Adjacency constraints:

- WA ≠ NT
- WA ≠ SA
- NT ≠ SA
- NT ≠ Q
- SA ≠ Q
- SA ≠ NSW
- SA ≠ V
- Q ≠ NSW
- NSW ≠ V

Tasmania `T` has no land-border constraint with the mainland states.

In [ ]:
VARIABLES = ["WA", "NT", "SA", "Q", "NSW", "V", "T"]
COLORS = ["Red", "Green", "Blue"]

DOMAINS = {
    var: list(COLORS)
    for var in VARIABLES
}

NEIGHBORS = {
    "WA": {"NT", "SA"},
    "NT": {"WA", "SA", "Q"},
    "SA": {"WA", "NT", "Q", "NSW", "V"},
    "Q": {"NT", "SA", "NSW"},
    "NSW": {"SA", "Q", "V"},
    "V": {"SA", "NSW"},
    "T": set(),
}

## Task 2.1 — Predict Consistency

Without running code, determine whether each partial assignment is consistent.

| Partial assignment | Consistent? | Violated constraint, if any |
|---|---|---|
| `{WA: Red, NT: Green}` |  |  |
| `{WA: Red, NT: Red}` |  |  |
| `{WA: Blue, SA: Blue}` |  |  |
| `{Q: Green, NSW: Red}` |  |  |
| `{SA: Green, V: Green}` |  |  |
| `{T: Red, WA: Red}` |  |  |

Then answer:

> Why is assigning `T=Red` and `WA=Red` not a problem?

**Your answer:**

## Task 2.2 — Write the Consistency Rule

Complete the sentence:

> Assigning color $c$ to region $X$ is consistent if, for every already-assigned neighbor $Y$ of $X$, ____________________________.

Then express the same idea as a logical condition.

**Your answer:**

# Part III — Implement the Consistency Test

Complete the function below.

It receives:

- `variable`: the variable we want to assign;
- `value`: the candidate color;
- `assignment`: the current partial assignment;
- `neighbors`: the constraint graph.

It should return `True` only if assigning the candidate value creates no conflict with already-assigned neighbors.

In [ ]:
def is_consistent(variable, value, assignment, neighbors):
    # TODO:
    # For each neighbor of 'variable':
    # if that neighbor is already assigned the same value,
    # return False.
    #
    # Otherwise return True.
    raise NotImplementedError

## Task 3.1 — Predict Before Testing

Before running the tests, predict the result of:

```python
assignment = {"WA": "Red", "NT": "Green"}
```

for:

- `is_consistent("SA", "Red", assignment, NEIGHBORS)`
- `is_consistent("SA", "Green", assignment, NEIGHBORS)`
- `is_consistent("SA", "Blue", assignment, NEIGHBORS)`

Explain each prediction.

**Your prediction:**

In [ ]:
assignment = {"WA": "Red", "NT": "Green"}

print("SA=Red  ->", is_consistent("SA", "Red", assignment, NEIGHBORS))
print("SA=Green->", is_consistent("SA", "Green", assignment, NEIGHBORS))
print("SA=Blue ->", is_consistent("SA", "Blue", assignment, NEIGHBORS))

assert is_consistent("SA", "Red", assignment, NEIGHBORS) is False
assert is_consistent("SA", "Green", assignment, NEIGHBORS) is False
assert is_consistent("SA", "Blue", assignment, NEIGHBORS) is True

print("Consistency tests passed.")

## Task 3.2 — Explain the Consistency Function

Answer:

1. Why do we only need to inspect **already-assigned** neighbors?
2. Why is comparing against every variable in the CSP unnecessary?
3. What information from $C$ is encoded in `NEIGHBORS`?
4. What would happen if the consistency function always returned `True`?

**Your answers:**

# Part IV — Manual Backtracking Trace

Use the fixed variable order:

```text
WA, NT, SA, Q, NSW, V, T
```

and fixed value order:

```text
Red, Green, Blue
```

Backtracking assigns one variable at a time.

If no value is consistent for the current variable, it returns to the previous variable and tries another value.

## Task 4.1 — Predict the First Assignments

Without running a solver, trace the first few decisions.

| Step | Variable | Tried value | Accept / Reject | Reason |
|---:|---|---|---|---|
| 1 | WA | Red |  |  |
| 2 | NT | Red |  |  |
| 3 | NT | Green |  |  |
| 4 | SA | Red |  |  |
| 5 | SA | Green |  |  |
| 6 | SA | Blue |  |  |

Then answer:

1. Why is `NT=Red` rejected after `WA=Red`?
2. Why is `SA=Blue` accepted after `WA=Red, NT=Green`?
3. What would the algorithm do if `SA` had no consistent color?

**Your answers:**

# Part V — Implement Backtracking Search

The solver will return:

```python
(solution, stats)
```

where `stats` contains:

- assignments tried;
- consistency failures;
- backtracks;
- maximum recursion depth;
- trace of decisions.

In [ ]:
def select_unassigned_variable(variables, assignment):
    for var in variables:
        if var not in assignment:
            return var
    return None

In [ ]:
def backtracking_search(variables, domains, neighbors):
    stats = {
        "assignments_tried": 0,
        "consistency_failures": 0,
        "backtracks": 0,
        "max_depth": 0,
        "trace": [],
    }

    def backtrack(assignment):
        depth = len(assignment)
        stats["max_depth"] = max(stats["max_depth"], depth)

        # TODO 1:
        # If assignment is complete, return a copy of it.

        # TODO 2:
        # Choose the next unassigned variable.
        variable = None

        # TODO 3:
        # For each value in domains[variable]:
        #   increment assignments_tried
        #   test consistency
        #   record the attempt in trace
        #
        # If consistent:
        #   assign variable=value
        #   recursively call backtrack
        #   if a solution is returned, return it
        #   otherwise remove the assignment and continue
        #
        # If inconsistent:
        #   increment consistency_failures

        # TODO 4:
        # If no value works, increment backtracks
        # and return None.
        pass

    solution = backtrack({})
    return solution, stats

## Task 5.1 — Predict Before Running

Before execution, predict:

- **First accepted assignment:**  
- **Second accepted assignment:**  
- **Third accepted assignment:**  
- **Do you expect at least one rejected color? Why?**  
- **Do you expect the solver to find a solution? Why?**

**Your prediction:**

In [ ]:
solution, stats = backtracking_search(
    VARIABLES,
    DOMAINS,
    NEIGHBORS
)

print("Solution:", solution)
print("Assignments tried:", stats["assignments_tried"])
print("Consistency failures:", stats["consistency_failures"])
print("Backtracks:", stats["backtracks"])
print("Maximum depth:", stats["max_depth"])

assert solution is not None
assert len(solution) == len(VARIABLES)

print("Backtracking solver returned a complete assignment.")

## Task 5.2 — Verify the Returned Solution

A solver should not be trusted only because it returned a dictionary.

Complete the checker below.

In [ ]:
def verify_solution(solution, variables, domains, neighbors):
    # TODO:
    # 1. Every variable must be assigned.
    # 2. Every assigned value must belong to that variable's domain.
    # 3. Every neighboring pair must have different values.
    raise NotImplementedError

In [ ]:
print("Solution valid:", verify_solution(
    solution,
    VARIABLES,
    DOMAINS,
    NEIGHBORS
))

assert verify_solution(solution, VARIABLES, DOMAINS, NEIGHBORS)
print("Solution verification passed.")

## Task 5.3 — Explain the Backtracking Algorithm

Answer in your own words.

1. What makes the search **depth-first**?
2. Why is an assignment removed after a recursive call fails?
3. What exactly is a **backtrack**?
4. Why can a consistent partial assignment still fail later?
5. Why does early constraint checking reduce unnecessary search?

**Your answers:**

# Part VI — Inspect the Search Trace

Print the first 20 recorded attempts.

In [ ]:
for row in stats["trace"][:20]:
    print(row)

## Task 6.1 — Interpret the Trace

Using the output:

1. Identify one accepted assignment.
2. Identify one rejected assignment.
3. State the constraint that caused the rejection.
4. If the solver backtracked, identify the variable involved.
5. How does the trace provide evidence that the solver is not simply assigning colors arbitrarily?

**Your answers:**

# Part VII — Variable Ordering Experiment

The order in which variables are selected can strongly affect search effort.

We first use the default order:

```text
WA, NT, SA, Q, NSW, V, T
```

Now compare it with an order that starts with the highly constrained central region `SA`:

```text
SA, NT, NSW, Q, WA, V, T
```

In [ ]:
ORDER_1 = ["WA", "NT", "SA", "Q", "NSW", "V", "T"]
ORDER_2 = ["SA", "NT", "NSW", "Q", "WA", "V", "T"]

solution_1, stats_1 = backtracking_search(ORDER_1, DOMAINS, NEIGHBORS)
solution_2, stats_2 = backtracking_search(ORDER_2, DOMAINS, NEIGHBORS)

print("ORDER 1")
print(" solution:", solution_1)
print(" assignments tried:", stats_1["assignments_tried"])
print(" failures:", stats_1["consistency_failures"])
print(" backtracks:", stats_1["backtracks"])

print("\nORDER 2")
print(" solution:", solution_2)
print(" assignments tried:", stats_2["assignments_tried"])
print(" failures:", stats_2["consistency_failures"])
print(" backtracks:", stats_2["backtracks"])

## Task 7.1 — Analyze Variable Ordering

Complete:

| Metric | Order 1 | Order 2 |
|---|---:|---:|
| Assignments tried |  |  |
| Consistency failures |  |  |
| Backtracks |  |  |

Then answer:

1. Did both orders find a valid solution?
2. Did they perform the same amount of search?
3. Why can variable order change search effort without changing the CSP itself?
4. Why might choosing a highly constrained variable earlier be useful?

**Your answers:**

# Part VIII — Model a Second CSP: Small Exam Scheduling

We now model a new problem using the same CSP ideas.

Variables:

```text
AI, DB, Networks, Security
```

Available slots:

```text
Mon-AM, Mon-PM, Tue-AM
```

Constraints:

- AI ≠ DB
- AI ≠ Security
- DB ≠ Networks
- Networks ≠ Security

In [ ]:
EXAM_VARIABLES = ["AI", "DB", "Networks", "Security"]
EXAM_SLOTS = ["Mon-AM", "Mon-PM", "Tue-AM"]

EXAM_DOMAINS = {
    exam: list(EXAM_SLOTS)
    for exam in EXAM_VARIABLES
}

EXAM_NEIGHBORS = {
    "AI": {"DB", "Security"},
    "DB": {"AI", "Networks"},
    "Networks": {"DB", "Security"},
    "Security": {"AI", "Networks"},
}

## Task 8.1 — Formal CSP Formulation

Write the exam-scheduling problem as

$$
(X,D,C).
$$

Be explicit:

- list every variable;
- state each domain;
- list each binary constraint.

**Your formulation:**

## Task 8.2 — Predict Before Solving

Before running backtracking:

1. Propose one complete schedule that you believe is valid.
2. Explain why `AI` and `DB` cannot share a slot.
3. Can `AI` and `Networks` share a slot under the given constraints?
4. Do you expect three slots to be sufficient? Why?

**Your prediction:**

In [ ]:
exam_solution, exam_stats = backtracking_search(
    EXAM_VARIABLES,
    EXAM_DOMAINS,
    EXAM_NEIGHBORS
)

print("Exam schedule:", exam_solution)
print("Assignments tried:", exam_stats["assignments_tried"])
print("Consistency failures:", exam_stats["consistency_failures"])
print("Backtracks:", exam_stats["backtracks"])

## Task 8.3 — Interpret the Exam Solution

1. Is the returned schedule valid?
2. Compare it with your manually proposed schedule.
3. Did the solver need to find the same valid schedule you predicted?
4. Why can a CSP have multiple valid solutions?
5. What would make this problem an **optimization problem** rather than only a satisfaction problem?

**Your answers:**

# Part IX — Debugging CSP Code

## Task 9.1 — Faulty Consistency Check

A student writes:

```python
def is_consistent(variable, value, assignment, neighbors):
    for other in assignment:
        if assignment[other] == value:
            return False
    return True
```

1. What is wrong conceptually?
2. Why does this reject assignments that may actually be legal?
3. What should be checked instead?

**Your answer:**

## Task 9.2 — Forgetting to Undo an Assignment

A student writes:

```python
assignment[var] = value
result = backtrack(assignment)

if result is not None:
    return result

# forgot: del assignment[var]
```

1. Why is deleting the failed assignment necessary?
2. What happens if it is not removed?
3. How does this affect later branches of the search?

**Your answer:**

## Task 9.3 — Declaring Success Too Early

A student returns the current assignment as soon as it is consistent, even if several variables are still unassigned.

1. Why is this incorrect?
2. What condition must be true before a CSP solution is returned?
3. Distinguish **consistent partial assignment** from **complete solution**.

**Your answer:**

# Part X — Personalized CSP Variation

Use the last digit of your student ID.

- `0–3`: `T` may use only `Red` or `Green`
- `4–6`: `WA` may use only `Green` or `Blue`
- `7–9`: `Q` may use only `Red` or `Blue`

Create a copy of the domain dictionary and modify only your assigned variable.

In [ ]:
LAST_DIGIT = None  # TODO: replace with an integer from 0 to 9

personal_domains = deepcopy(DOMAINS)

if LAST_DIGIT is not None:
    if 0 <= LAST_DIGIT <= 3:
        personal_domains["T"] = ["Red", "Green"]
        PERSONAL_CHANGE = "T -> {Red, Green}"
    elif 4 <= LAST_DIGIT <= 6:
        personal_domains["WA"] = ["Green", "Blue"]
        PERSONAL_CHANGE = "WA -> {Green, Blue}"
    elif 7 <= LAST_DIGIT <= 9:
        personal_domains["Q"] = ["Red", "Blue"]
        PERSONAL_CHANGE = "Q -> {Red, Blue}"
    else:
        raise ValueError("LAST_DIGIT must be between 0 and 9")

    print("Assigned variation:", PERSONAL_CHANGE)

## Task 10.1 — Predict Before Running

Write:

- **My domain modification:**  
- **Which CSP component changed: $X$, $D$, or $C$?**  
- **Do I expect the problem to remain solvable? Why?**  
- **Do I expect the number of assignments tried to change? Why?**

**Your prediction:**

In [ ]:
if LAST_DIGIT is not None:
    personal_solution, personal_stats = backtracking_search(
        VARIABLES,
        personal_domains,
        NEIGHBORS
    )

    print("Personal solution:", personal_solution)
    print("Assignments tried:", personal_stats["assignments_tried"])
    print("Consistency failures:", personal_stats["consistency_failures"])
    print("Backtracks:", personal_stats["backtracks"])

## Task 10.2 — Explain the Personalized Result

1. Was the CSP still solvable?
2. Did the returned solution differ from the original?
3. Did the search effort change?
4. Why can reducing a domain sometimes make search easier?
5. Why can reducing a domain also make a CSP harder or even unsatisfiable?

**Your answers:**

# Part XI — Individual Understanding Check

Your instructor may ask one short question about your notebook.

Possible prompts:

- Show me where the constraint check occurs.
- What is the difference between a domain and an assignment?
- Why do we remove a value after a recursive branch fails?
- Why can a consistent partial assignment still lead to failure?
- What exactly counts as a backtrack?
- Why did changing variable order affect search effort?
- Which component changed in your personalized CSP?
- Why is `T` independent of the mainland constraints?

> You are expected to explain the **AI concept represented by the code**, not memorize Python syntax.

# Reflection

Answer concisely but precisely.

### R1 — CSP Representation
Why is separating variables, domains, and constraints useful?

**Answer:**

### R2 — Backtracking
Why is backtracking more efficient than generating every complete assignment and checking constraints only at the end?

**Answer:**

### R3 — Partial Assignments
What is the difference between a consistent partial assignment and a solution?

**Answer:**

### R4 — Variable Ordering
Why can variable order affect runtime even though it does not change the set of valid solutions?

**Answer:**

### R5 — Looking Ahead
What additional mechanism could reduce domains **before** deeper recursion rather than waiting for inconsistency later?

**Answer:**

# Submission Checklist

Before submitting, verify that your notebook contains:

- [ ] CSP component identification;
- [ ] partial-vs-complete assignment reasoning;
- [ ] consistency predictions;
- [ ] working `is_consistent`;
- [ ] manual backtracking trace;
- [ ] working backtracking solver;
- [ ] working solution verifier;
- [ ] search-trace interpretation;
- [ ] variable-ordering experiment;
- [ ] second CSP formulation and solution;
- [ ] debugging answers;
- [ ] personalized domain variation;
- [ ] prediction before the personalized run;
- [ ] reflection answers;
- [ ] visible outputs from important code cells.

Suggested filename:

```text
Lab07_StudentID.ipynb
```

# Assessment Guide — 10 Marks

| Component | Marks | Evidence expected |
|---|---:|---|
| **Correct implementation** | **2.0** | Consistency test, backtracking solver, and verifier work correctly |
| **Algorithmic / modeling justification** | **3.0** | Correctly explains $(X,D,C)$, consistency, recursion, backtracking, and modeling choices |
| **Experimental analysis** | **2.0** | Interprets trace, variable ordering, and second-CSP results |
| **Trace / prediction / debugging** | **1.0** | Manual trace, predictions, and diagnosis of faulty CSP logic |
| **Individual understanding check** | **1.0** | Short explanation of selected part of the student's own work |
| **Code quality & completeness** | **1.0** | Readable code, complete responses, required outputs |
| **Total** | **10.0** |  |

> **Key rule:** Correct code without adequate explanation earns only a limited portion of the marks.

## Key Takeaways

- A CSP is represented by

$$
(X,D,C).
$$

- A partial assignment can be consistent without yet being a complete solution.
- Backtracking assigns one variable at a time and abandons inconsistent branches.
- Constraint checking should happen as early as possible.
- Variable ordering can greatly affect search effort.
- The underlying set of valid solutions may remain unchanged even when the search order changes.
- A generic CSP framework can model map coloring, scheduling, puzzles, and many other problems.

The next lab will extend CSP solving with **MRV, degree heuristic, least-constraining value, and forward checking**.